In [64]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from scipy import stats as st
import os
import csv
from collections import Counter
from difflib import SequenceMatcher

In [65]:
def string_similarity_match(str1, str2, threshold=90):
    # Calculate similarity ratio using SequenceMatcher
    ratio = SequenceMatcher(None, str1.lower(), str2.lower()).ratio()
    similarity_percentage = ratio * 100 
    
    return similarity_percentage >= threshold

def get_best_match_index(target_string, candidate_list, threshold=90):
    if not candidate_list:
        return None
    
    best_similarity = 0.0
    best_index = None
    
    for i, candidate in enumerate(candidate_list):
        ratio = SequenceMatcher(None, str(target_string).lower(), str(candidate).lower()).ratio()
        similarity = ratio * 100
        
        if similarity > best_similarity:
            best_similarity = similarity
            best_index = i
    
    # Return index only if it meets threshold
    if best_similarity >= threshold:
        print('best sim:', best_similarity)
    return best_index if best_similarity >= threshold else None

In [66]:
def transform_mendeley_dict(original_dict):
    
    transformed_dict = {'message': [], 'label': []}
    
    # Process ham messages (label = 0)
    for message in original_dict['ham']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('ham')
    
    # Process spam messages (label = 1)
    for message in original_dict['spam']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('spam')
    
    return pd.DataFrame(transformed_dict)

In [67]:
def read_file(file_path):
    with open(file_path, 'r') as file:
        content = file.readlines()
    return content

In [68]:
mendeley_dataset = pd.read_csv('../Dataset/Mendeley_dataset_5971.csv')
mendeley_dataset['LABEL'] = mendeley_dataset['LABEL'].replace('Smishing', 'smishing')
mendeley_dataset['LABEL'] = mendeley_dataset['LABEL'].replace('Spam', 'spam')
mendeley_dataset.head()

,LABEL,TEXT,URL,EMAIL,PHONE
0,ham,Your opinion about me? 1. Over 2. Jada 3. Kusr...,No,No,No
1,ham,What's up? Do you want me to come online? If y...,No,No,No
2,ham,So u workin overtime nigpun?,No,No,No
3,ham,"Also sir, i sent you an email about how to log...",No,No,No
4,smishing,Please Stay At Home. To encourage the notion o...,No,No,No


In [69]:
# mendeley_dataset = transform_mendeley_dict(mendeley_dataset)
# mendeley_dataset.head()

In [70]:
# Counter(pd.read_csv('../Dataset/URL Data/'+'mendeley Dataset_'+'.csv')['URL'].to_list())

In [71]:
mendeley_dataset['Extracted URL'] = pd.read_csv('../Dataset/URL Data/'+'Mendeley Dataset_'+'.csv')['URL']
mendeley_dataset['Message Len'] = [len(i) for i in mendeley_dataset['TEXT']]
mendeley_dataset.head()

,LABEL,TEXT,URL,EMAIL,PHONE,Extracted URL,Message Len
0,ham,Your opinion about me? 1. Over 2. Jada 3. Kusr...,No,No,No,NaN,136
1,ham,What's up? Do you want me to come online? If y...,No,No,No,NaN,79
2,ham,So u workin overtime nigpun?,No,No,No,NaN,28
3,ham,"Also sir, i sent you an email about how to log...",No,No,No,NaN,173
4,smishing,Please Stay At Home. To encourage the notion o...,No,No,No,NaN,152


In [72]:
mendeley_website_analysis_data = pd.read_csv('../Dataset/URL Data/'+'Mendeley Websites Analysis'+'.csv')
mendeley_website_analysis_data = mendeley_website_analysis_data.drop(columns=['spam', 'smishing'])
mendeley_website_analysis_data.head()

,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,www.07781482378.com,www.07781482378.com,0,0,-1,0
1,https://m.paytm.me/fastag-help,m.paytm.me,1688,23,200,0
2,http://store.ovi.com/publisher/F-Secure/?cid=f,store.ovi.com,0,0,-1,0
3,http://www.e-tlp.co.uk/reward,www.e-tlp.co.uk,0,0,-1,0
4,http://careers.com,careers.com,64328,243,200,0


In [73]:
string_similarity_match('http://wiseschool.com','wiseschool.com')

False

In [74]:
mendeley_website_analysis_data.iloc[0][0]

C:\Users\mmia43\AppData\Local\Temp\ipykernel_10296\2547819133.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  mendeley_website_analysis_data.iloc[0][0]


'www.07781482378.com'

In [75]:
# for row in mendeley_website_analysis_data.itertuples(index=False):
#     print(row[0])

In [76]:
import tldextract

def FQDN(Url):
    
    if type(Url) !=str:
        return ''

    url_extract_res = tldextract.extract(Url)
    fqdn = ''
    if url_extract_res.subdomain:
        fqdn = url_extract_res.subdomain + '.' + url_extract_res.domain + '.' + url_extract_res.suffix
        # fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    else:
        fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    
    return fqdn

In [77]:
extracted_urls = mendeley_dataset['Extracted URL'].values
extracted_urls_fqdn = [FQDN(i) for i in extracted_urls]

# Create a dictionary for O(1) lookup instead of O(n) loop
fqdn_to_index = {fqdn: idx for idx, fqdn in enumerate(mendeley_website_analysis_data['FQDN'])}
website_data = mendeley_website_analysis_data.iloc[:, 1:6].values

# Pre-allocate lists for better performance
n_urls = len(extracted_urls)
fqdn = extracted_urls_fqdn
website_size = [''] * n_urls
text_content_len = [''] * n_urls
status_code = [''] * n_urls
parked = [''] * n_urls

# Single optimized loop with dictionary lookup
for i, url in enumerate(extracted_urls):
    if url and extracted_urls_fqdn[i]:  # Check both url and fqdn exist
        matched_idx = fqdn_to_index.get(extracted_urls_fqdn[i])  # O(1) lookup
        if matched_idx is not None:
            row_data = website_data[matched_idx]
            website_size[i] = row_data[1]
            text_content_len[i] = row_data[2]
            status_code[i] = row_data[3]
            parked[i] = row_data[4]

In [78]:
mendeley_dataset['FQDN'] = fqdn
mendeley_dataset['Website Size in KB'] = website_size
mendeley_dataset['Website Textual Content Length'] = text_content_len
mendeley_dataset['Status Code'] = status_code
mendeley_dataset['Parked'] = parked

In [79]:
mendeley_dataset = mendeley_dataset.replace('', np.nan)
mendeley_dataset.head()

C:\Users\mmia43\AppData\Local\Temp\ipykernel_10296\3879617348.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mendeley_dataset = mendeley_dataset.replace('', np.nan)


,LABEL,TEXT,URL,EMAIL,PHONE,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,ham,Your opinion about me? 1. Over 2. Jada 3. Kusr...,No,No,No,NaN,136,NaN,NaN,NaN,NaN,NaN
1,ham,What's up? Do you want me to come online? If y...,No,No,No,NaN,79,NaN,NaN,NaN,NaN,NaN
2,ham,So u workin overtime nigpun?,No,No,No,NaN,28,NaN,NaN,NaN,NaN,NaN
3,ham,"Also sir, i sent you an email about how to log...",No,No,No,NaN,173,NaN,NaN,NaN,NaN,NaN
4,smishing,Please Stay At Home. To encourage the notion o...,No,No,No,NaN,152,NaN,NaN,NaN,NaN,NaN


In [80]:
# Counter(mendeley_dataset['FQDN'].to_list())
mendeley_dataset[(mendeley_dataset['Extracted URL'].notna()) & (mendeley_dataset['FQDN'].isna())]

,LABEL,TEXT,URL,EMAIL,PHONE,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked


In [81]:
# Counter(mendeley_dataset['FQDN'].to_list())
mendeley_dataset[(mendeley_dataset['Extracted URL'].notna()) & (mendeley_dataset['FQDN'].notna())]

,LABEL,TEXT,URL,EMAIL,PHONE,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
5,smishing,BankOfAmerica Alert 137943. Please follow http...,yes,No,yes,http://bit.do/cgjK-and,76,bit.do,0.0,0.0,-1.0,0.0
23,spam,lyricalladie(21/F) is inviting you to be her f...,No,No,yes,www.SMS.ac/u/hmmross,137,www.SMS.ac,1034.0,0.0,200.0,1.0
27,spam,Hello from Orange. For 1 month's free access t...,yes,No,No,www.orange.co.uk/ow,156,www.orange.co.uk,3746.0,672.0,200.0,0.0
48,smishing,Apple ID: [BUXCX7GBVwWCcOD Final Notification ...,yes,No,No,http://verifyapple.uk,159,verifyapple.uk,0.0,0.0,-1.0,0.0
86,spam,HMV BONUS SPECIAL 500 pounds of genuine HMV vo...,yes,No,yes,www.100percent-real.com,145,www.100percent-real.com,0.0,0.0,-1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
5687,spam,Knock Knock Txt whose there to 80082 to enter ...,yes,No,yes,www.tkls.com,154,www.tkls.com,114.0,0.0,200.0,1.0
5704,spam,Our Stock In News Call : Mcleod Russel Earns I...,yes,No,No,www.StockInNews.com,149,www.StockInNews.com,821.0,0.0,200.0,1.0
5808,smishing,\tReply with your name and address and YOU WIL...,yes,No,yes,www.phb1.com,160,www.phb1.com,157787.0,2394.0,200.0,0.0
5827,smishing,Natalie (20/F) is inviting you to be her frien...,yes,No,No,www.SMS.ac/u/natalie2k9,105,www.SMS.ac,1034.0,0.0,200.0,1.0


In [82]:
mendeley_dataset['LABEL'].value_counts()

LABEL
ham         4844
smishing     638
spam         489
Name: count, dtype: int64

In [83]:
print(len(mendeley_dataset))

5971


In [84]:
#messages with URL
print(len(mendeley_dataset[(mendeley_dataset['Extracted URL'].notna())]), len(mendeley_dataset[(mendeley_dataset['Extracted URL'].notna())])/len(mendeley_dataset))

172 0.02880589515993971


In [85]:
#spam messages with URL
print(len(mendeley_dataset[(mendeley_dataset['Extracted URL'].notna()) & (mendeley_dataset['LABEL']=='spam')]), len(mendeley_dataset[(mendeley_dataset['Extracted URL'].notna()) & (mendeley_dataset['LABEL']=='spam')])/len(mendeley_dataset[mendeley_dataset['LABEL']=='spam']))

73 0.1492842535787321


In [86]:
#smishing messages with URL
print(len(mendeley_dataset[(mendeley_dataset['Extracted URL'].notna()) & (mendeley_dataset['LABEL']=='smishing')]), len(mendeley_dataset[(mendeley_dataset['Extracted URL'].notna()) & (mendeley_dataset['LABEL']=='smishing')])/len(mendeley_dataset[mendeley_dataset['LABEL']=='smishing']))

97 0.15203761755485892


In [87]:
#unique FQDN
len(set(mendeley_dataset[(mendeley_dataset['FQDN'].notna())]['FQDN']))

96

In [88]:
only_unique_live_websites_data = mendeley_dataset.drop_duplicates(subset=['FQDN'])
#live websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200)]))

#live websites ham
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200) & (only_unique_live_websites_data['LABEL']=='ham')]))

#live websites spam
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200) & (only_unique_live_websites_data['LABEL']=='spam')]))

#live websites spam
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200) & (only_unique_live_websites_data['LABEL']=='smishing')]))

22
0
12
10


In [89]:
#parked websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1)]))

#parked websites ham
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1) & (only_unique_live_websites_data['LABEL']=='ham')]))

#parked websites spam
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1) & (only_unique_live_websites_data['LABEL']=='spam')]))

#parked websites spam
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1) & (only_unique_live_websites_data['LABEL']=='smishing')]))

11
0
8
3


In [90]:
mendeley_dataset.to_csv('../Dataset/Refined_Mendeley_Smishing_Dataset.csv', index=None)